# Evaluate Gemini 2.5 Flash using MedEvidence Data 

Only evaluate the questions that do not need full text


In [31]:
from utils import render_prompt
import pandas as pd
from models.gemini import Gemini
from utils import load_json_file, save_dataset_to_json
import seaborn as sns
import matplotlib.pyplot as plt

EVAL_MODEL_TEMPERATURE = 0.0
EVIDENCE_DIRECTION_PROMPT_TEMPLATE_NAMES = "evidence_direction_question"

## Submit Job to Gemini

In [15]:
# load medevidence dataset
medevidence_file_path = "../data/medEvidence.csv"
medevidence_df = pd.read_csv(medevidence_file_path)

medevidence_df.shape

(284, 11)

In [16]:
# filter out rows where fulltext_required is yes. Only keep rows where fulltext_required is no
filtered_medevidence_df = medevidence_df[medevidence_df['fulltext_required'] == 'no']
filtered_medevidence_df.shape

(216, 11)

In [17]:
# format data for model evaluation
data = filtered_medevidence_df.to_dict(orient='records')
formatted_input_for_model_evaluator = {}
for row in data:
    question = row['question']
    sources = row['sources']
    question_id = row['question_id']

    eval_direction_input = render_prompt(EVIDENCE_DIRECTION_PROMPT_TEMPLATE_NAMES, template_dir="./prompts", question=question, context=sources)
    formatted_input_for_model_evaluator[f"{question_id}"] = eval_direction_input

In [ ]:
# load the evaluation model
eval_model = Gemini("2.5")

# eval_model.submit_batch(formatted_input_for_model_evaluator, EVAL_MODEL_TEMPERATURE)

Uploaded batch input file: files/6ew2xtack4tk
Created batch: batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y


'batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y'

## Get results

In [90]:
from google import genai
from dotenv import load_dotenv
import os
import json

load_dotenv(override=True)
gemini_api_key = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=gemini_api_key)

In [91]:
BATCH_ID = "batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y"

In [100]:
print(f"Retrieving status for job: {BATCH_ID}")

batch_job = gemini_client.batches.get(name=BATCH_ID)
print(f"Current state: {batch_job.state.name}")
print(f"Job details: {batch_job}")

Retrieving status for job: batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y
Current state: JOB_STATE_RUNNING
Job details: name='batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y' display_name=None state=<JobState.JOB_STATE_RUNNING: 'JOB_STATE_RUNNING'> error=None create_time=datetime.datetime(2026, 2, 24, 18, 47, 38, 274769, tzinfo=TzInfo(UTC)) start_time=None end_time=None update_time=datetime.datetime(2026, 2, 24, 18, 52, 8, 224752, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=None completion_stats=None


In [99]:
batch_jobs = gemini_client.batches.list()

# Optional query config:
# batch_jobs = client.batches.list(config={'page_size': 5})

for batch_job in batch_jobs:
    print(batch_job)


name='batches/w65gdu7qkpwlfwjg0vob4ptkvik7ki3d3n4y' display_name=None state=<JobState.JOB_STATE_RUNNING: 'JOB_STATE_RUNNING'> error=None create_time=datetime.datetime(2026, 2, 24, 18, 47, 38, 274769, tzinfo=TzInfo(UTC)) start_time=None end_time=None update_time=datetime.datetime(2026, 2, 24, 18, 52, 8, 224752, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=None completion_stats=None
name='batches/jy8bq5pu3sdc4t3oquabytams3jeumjjqt6v' display_name=None state=<JobState.JOB_STATE_SUCCEEDED: 'JOB_STATE_SUCCEEDED'> error=None create_time=datetime.datetime(2026, 2, 13, 2, 10, 8, 717797, tzinfo=TzInfo(UTC)) start_time=None end_time=datetime.datetime(2026, 2, 13, 3, 24, 11, 104526, tzinfo=TzInfo(UTC)) update_time=datetime.datetime(2026, 2, 13, 3, 24, 11, 104526, tzinfo=TzInfo(UTC)) model='models/gemini-2.5-flash' src=None dest=BatchJobDestination(
  file_name='files/batch-jy8bq5pu3sdc4t3oquabytams3jeumjjqt6v'
) completion_stats=None
name='batches/5fy46pj9ilcop3vuefkj5drufi8h

In [22]:
def get_batch_results(batch_name: str) -> dict:
    """
    Fetches batch status and outputs (if completed)
    """
    batch_job = gemini_client.batches.get(name=batch_name)

    # The output is in another file.
    result_file_name = batch_job.dest.file_name
    print(f"Results are in file: {result_file_name}")

    print("\nDownloading and parsing result file content...")
    file_content_bytes = gemini_client.files.download(file=result_file_name)
    file_content = file_content_bytes.decode('utf-8')

    results = {}
    for line in file_content.splitlines():
        if line:
            record = json.loads(line)
            custom_id = record["key"]
            response = record["response"]["candidates"][0]
            output_text = response["content"]["parts"][0]["text"]
            results[custom_id] = output_text
    return results

In [23]:
results = get_batch_results(BATCH_ID)

save_dataset_to_json(results, "../data/medevidence_evaluation_results.json")

Results are in file: files/batch-jy8bq5pu3sdc4t3oquabytams3jeumjjqt6v



## Analyze Result

In [24]:
results = load_json_file("../data/medevidence_evaluation_results.json")

print(len(results))

640


In [ ]:
medevidence_file_path = "../data/medEvidence.csv"
medevidence_df = pd.read_csv(medevidence_file_path)
filtered_medevidence_df = medevidence_df[medevidence_df['fulltext_required'] == 'no']
data = filtered_medevidence_df.to_dict(orient='records')

In [ ]:
def extract_answer(text: str) -> str:
    """
    Extracts only nswer if answer exists else return the full text

    :param text: string of text

    :return: str
    """
    delimiter = "**Answer**:"
    parts = text.split(delimiter, 1)

    # If len is > 1, delimiter existed; return second part, stripped of whitespace
    if len(parts) > 1:
        return parts[1].strip()

    return text.lower()

In [ ]:
for row in data:
    question_id = row['question_id']
    if question_id in results:
        response = results[f"req-{question_id}"]
        row["gemini_output"] = extract_answer(response)

In [ ]:
# % of valid gemini output
# the output must be one of the following: higher, lower, no difference, insufficient data, uncertain effect
valid_outputs = {"higher", "lower", "no difference", "insufficient data", "uncertain effect"}
valid_count = 0
for row in data:
    if "gemini_output" in row and row["gemini_output"] in valid_outputs:
        valid_count += 1

print(f"Valid outputs: {valid_count}/{len(data)} ({valid_count/len(data)*100:.2f}%)")

In [ ]:
# % of accuracy of gemini output compared to the human answer which is the "answer" key in the dataset
accurate_count = 0
for row in data:
    if "gemini_output" in row and row["gemini_output"] == row["answer"]:
        accurate_count += 1
        
print(f"Accurate outputs: {accurate_count}/{len(data)} ({accurate_count/len(data)*100:.2f}%)")

In [ ]:
# get the recall by each answer category
answer_categories = valid_outputs
matrix_by_category = {category: {"tp": 0, "fn": 0} for category in answer_categories}
for row in data:
    if "gemini_output" in row and row["answer"] in answer_categories:
        human_answer = row["answer"]
        if row["gemini_output"] == human_answer:
            matrix_by_category[human_answer]["tp"] += 1
        else:
            matrix_by_category[human_answer]["fn"] += 1

recall_data = []
for category, counts in matrix_by_category.items():
    tp = counts["tp"]
    fn = counts["fn"]
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_data.append({"category": category, "recall": recall})
    print(f"Category: {category}, Recall: {recall:.2f} ({tp}/{tp + fn})")

In [ ]:
output_counts = {category: 0 for category in valid_outputs}
for row in data:
    if "gemini_output" in row and row["gemini_output"] in valid_outputs:
        output_counts[row["gemini_output"]] += 1

plt.figure(figsize=(10, 6))
sns.barplot(x=list(output_counts.keys()), y=list(output_counts.values()))
plt.title("Distribution of Gemini Outputs")
plt.xlabel("Gemini Output Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
recall_df = pd.DataFrame(recall_data)
plt.figure(figsize=(10, 6))
sns.barplot(x="category", y="recall", data=recall_df)
plt.title("Recall by Answer Category")
plt.xlabel("Answer Category")
plt.ylabel("Recall")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# seaborn bar plot of accuracy over time based on review_year

accuracy_by_year = {}
for row in data:
    if "gemini_output" in row and row["answer"] in valid_outputs:
        review_year = row["review_year"]
        if review_year not in accuracy_by_year:
            accuracy_by_year[review_year] = {"tp": 0, "total": 0}
        accuracy_by_year[review_year]["total"] += 1
        if row["gemini_output"] == row["answer"]:
            accuracy_by_year[review_year]["tp"] += 1

results_df = pd.DataFrame([
    {"review_year": year, "accuracy": counts["tp"] / counts["total"] if counts["total"] > 0 else 0.0, "model": "gemini-2.5"}
     for year, counts in accuracy_by_year.items()
])

sns.barplot(x='review_year', y='accuracy', hue='model', data=results_df)
plt.title('Accuracy Over Time by Model')
plt.xlabel('Review Year')
plt.ylabel('Accuracy')
plt.show()